# Exercise 1 - 3D - 2D

In [1]:
import numpy as np
import cv2 as cv2
from numpy.linalg import inv, pinv
import matplotlib.pyplot as plt

Recall from the slides the steps from Algorithm 3:

![title](algorithm_3.png)

![title](PnP.png)

# Exercise 1a)
The steps 1)-2.1) has already been done, and is saved in corresponding files. The exercise is to implement step 2.2) by filling in the missing code below

In [7]:
def featureTracking(prev_img, next_img, prev_points, world_points):


    # Parameters for optical flow
    params = dict(
        winSize=(21, 21),
        maxLevel=3,
        criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01)
    )

    # Perform feature tracking (Lucas–Kanade optical flow)
    next_points, status, _ = cv2.calcOpticalFlowPyrLK(
        prev_img, next_img, prev_points, None, **params
    )

    # Keep only successfully tracked points (status == 1)
    status = status.reshape(-1)
    valid_idx = (status == 1)

    world_points = world_points[valid_idx]
    prev_points = prev_points[valid_idx]
    next_points = next_points[valid_idx]

    return world_points, prev_points, next_points

Hint: Exercise 4 in week 2

# Exercise 1b)
Continue the algorithm by implementing step 2.3)

In [6]:
# Camera intrinsics (from your notebook)
K = np.array([[7.188560e+02, 0.000000e+00, 6.071928e+02],  # camera matrix
              [0, 7.188560e+02, 1.852157e+02],
              [0, 0, 1]])

# First (reference) image
reference_img = np.load("img_" + str(0) + ".npy")

for t in range(1, 6):

    # the image at current time=t
    curImage = np.load("img_" + str(t) + ".npy")
    # the 3D landmarks in the world coordinates which have been computed in time=t-1
    landmark_3D = np.load("landmark_" + str(t-1) + ".npy")
    # the 2D coordinates of the 3D points in the previous frame at time=t-1
    reference_2D = np.load("reference_2D_" + str(t-1) + ".npy")
    
    # the 2D landmarks at the current time = t
    landmark_3D, reference_2D, tracked_2Dpoints = featureTracking(reference_img, 
                                                                  curImage, 
                                                                  reference_2D,
                                                                  landmark_3D)
    
    """
    Using OpenCV, implement PnP using Ransac
    """
    # Ensure proper shapes/dtypes for solvePnPRansac
    # tracked_2Dpoints: (N,1,2) or (N,2)
    if tracked_2Dpoints.ndim == 2:  # (N,2) -> (N,1,2)
        tracked_2Dpoints = tracked_2Dpoints.reshape(-1, 1, 2).astype(np.float64)
    else:
        tracked_2Dpoints = tracked_2Dpoints.astype(np.float64)
    tracked_2Dpoints = np.ascontiguousarray(tracked_2Dpoints)

    landmark_3D = landmark_3D.astype(np.float64)
    landmark_3D = np.ascontiguousarray(landmark_3D)

    # --- PnP with RANSAC ---
    if len(landmark_3D) >= 6:
        ok, rvec, tvec, inliers = cv2.solvePnPRansac(
            objectPoints=landmark_3D,          # (N,3)
            imagePoints=tracked_2Dpoints,      # (N,1,2)
            cameraMatrix=K.astype(np.float64),
            distCoeffs=None,
            flags=cv2.SOLVEPNP_ITERATIVE,
            reprojectionError=8.0,
            iterationsCount=100
        )
        if not ok:
            raise RuntimeError("solvePnPRansac failed to estimate a pose.")
    else:
        raise ValueError(f"Not enough correspondences after tracking: {len(landmark_3D)} found, need >= 6.")

    """
    Transform the translation and rotation into the world frame
    """
    # rvec/tvec describe camera pose w.r.t. world: [R_cw | t_cw]
    R_cw, _ = cv2.Rodrigues(rvec)        # (3x3)
    t_cw = tvec.reshape(3, 1)            # (3x1)

    # Convert to world-frame pose of the camera: [R_w_c | t_w_c]
    R_w_c = R_cw.T
    t_w_c = -R_cw.T @ t_cw

    # (Optional) rotation back to axis-angle for logging
    rvec_w, _ = cv2.Rodrigues(R_w_c)
    
    print(t_w_c[0], t_w_c[1], t_w_c[2], rvec_w[0], rvec_w[1], rvec_w[2])

    # update for next timestep
    reference_img = curImage


[-0.00110282] [-0.00067164] [-0.00078343] [7.40069214e-05] [7.35119065e-05] [-9.8454428e-05]
[-0.00363949] [-0.00875088] [0.67580836] [0.00216658] [-0.00325854] [0.00244333]
[-0.01096317] [-0.01635688] [1.37740874] [0.00364614] [-0.00751509] [0.00099692]
[-0.03156638] [-0.02560108] [2.09967983] [0.00509583] [-0.01121646] [0.00082978]
[-0.04971864] [-0.03532535] [2.83300707] [0.00561424] [-0.0161333] [-0.00041981]


Hint: The output should look similar to:

[-0.00110282] [-0.00067164] [-0.00078343] [-7.40069212e-05] [-7.35119065e-05] [9.84544279e-05]

[-0.00363946] [-0.00875075] [0.67580842] [-0.0021666] [0.00325853] [-0.00244333]

[-0.01096271] [-0.01635663] [1.3774094] [-0.00364615] [0.0075151] [-0.00099691]

[-0.0315663] [-0.02560111] [2.0996797] [-0.00509583] [0.01121646] [-0.00082978]

[-0.04971858] [-0.03532535] [2.8330071] [-0.00561424] [0.0161333] [0.00041981]

# Exercise 1c)
What approximate direction did the camera move in?

In [8]:
# If you printed/collected world-frame translations each step as t_w_c:
# e.g., inside the loop do: traj.append(t_w_c.ravel())
# Here we just analyze them:

import numpy as np

traj = np.array([
    [-0.00363946, -0.00875075, 0.67580842],
    [-0.01096271, -0.01635663, 1.37740940],
    [-0.03156630, -0.02560111, 2.09967970],
    [-0.04971858, -0.03532535, 2.83300710],
], dtype=float)

net = traj[-1] - traj[0]
unit_dir = net / np.linalg.norm(net)

print("Net displacement (Δx, Δy, Δz):", net)
print("Unit direction:", unit_dir)

# Simple verbal interpretation
print(
    "Direction: forward (+Z), slight left (−X), slight down (−Y). "
    f"Δ = ({net[0]:+.3f}, {net[1]:+.3f}, {net[2]:+.3f})"
)


Net displacement (Δx, Δy, Δz): [-0.04607912 -0.0265746   2.15719868]
Unit direction: [-0.02135414 -0.01231529  0.99969612]
Direction: forward (+Z), slight left (−X), slight down (−Y). Δ = (-0.046, -0.027, +2.157)
